# Pearls AQI Predictor — Exploratory Data Analysis
Exploring trends in the collected Karachi AQI dataset from Hopsworks.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv("../.env")

from src.hopsworks_client import connect, FEATURE_GROUP_NAME, FEATURE_GROUP_VERSION

project = connect()
fs = project.get_feature_store()
fg = fs.get_feature_group(name=FEATURE_GROUP_NAME, version=FEATURE_GROUP_VERSION)
df = fg.read()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
print(f"{len(df)} rows loaded")
df.head()

## AQI over time

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["timestamp"], df["aqi"], linewidth=1)
plt.title("Karachi AQI over time")
plt.xlabel("Date")
plt.ylabel("AQI")
plt.tight_layout()
plt.show()

## AQI by hour of day
Checking whether pollution follows a daily rhythm (e.g. traffic-driven peaks).

In [ ]:
hourly_avg = df.groupby("hour")["aqi"].mean()
plt.figure(figsize=(10, 4))
hourly_avg.plot(kind="bar", color="steelblue")
plt.title("Average AQI by hour of day")
plt.xlabel("Hour (UTC)")
plt.ylabel("Average AQI")
plt.tight_layout()
plt.show()

## Correlation between pollutants, weather, and AQI

In [ ]:
cols = ["aqi", "pm2_5", "pm10", "co", "no2", "so2", "o3", "temp", "humidity", "pressure", "wind_speed"]
corr = df[cols].corr()
plt.figure(figsize=(9, 7))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(cols)), cols, rotation=45, ha="right")
plt.yticks(range(len(cols)), cols)
plt.title("Correlation matrix: pollutants, weather, and AQI")
plt.tight_layout()
plt.show()

## AQI category distribution

In [ ]:
category_counts = df["aqi_category"].value_counts()
plt.figure(figsize=(8, 4))
category_counts.plot(kind="bar", color="darkorange")
plt.title("Distribution of AQI categories in the dataset")
plt.ylabel("Number of readings")
plt.tight_layout()
plt.show()
category_counts